<a href="https://colab.research.google.com/github/AprilArn/3D-visualization-based-on-web/blob/main/model_itsd_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Indonesian Traffic Sign Detection - YOLOv8

I'm working with 21 classes of traffic signs in Indonesia

In [ ]:
from google.colab import drive
drive.mount( '/content/drive' )

In [ ]:
# Install YOLO package from ultralytics
!pip install ultralytics

In [ ]:
import numpy as np
import pandas as pd
import os
import shutil
import cv2
import random
import glob
from math import radians, sin, cos
from PIL import Image, ImageEnhance
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg
import shutil
import re

from ultralytics import YOLO

In [ ]:
# Original dataset path
dataset_path = '/content/drive/MyDrive/Datasets/indonesian-traffic-sign'
train_path = os.path.join( dataset_path, 'train' )
test_path = os.path.join( dataset_path, 'test' )

# Create new folder for modified folder structure
# Modified dataset path
MODIFIED_PATH = os.path.join( dataset_path, 'modified' )
os.makedirs( MODIFIED_PATH, exist_ok=True )

# Create new folder for images and labels in modified folder
IMAGES_PATH = os.path.join( MODIFIED_PATH, 'images' )
LABELS_PATH = os.path.join( MODIFIED_PATH, 'labels' )
os.makedirs( IMAGES_PATH, exist_ok=True )
os.makedirs( LABELS_PATH, exist_ok=True )

# Create new folder for train, valid, test in the images and labels folder
TRAIN_IMAGES_PATH = os.path.join( IMAGES_PATH, 'train' )
TRAIN_LABELS_PATH = os.path.join( LABELS_PATH, 'train' )
VALID_IMAGES_PATH = os.path.join( IMAGES_PATH, 'valid' )
VALID_LABELS_PATH = os.path.join( LABELS_PATH, 'valid' )
TEST_IMAGES_PATH = os.path.join( IMAGES_PATH, 'test' )
TEST_LABELS_PATH = os.path.join( LABELS_PATH, 'test' )
os.makedirs( TRAIN_IMAGES_PATH, exist_ok=True )
os.makedirs( TRAIN_LABELS_PATH, exist_ok=True )
os.makedirs( VALID_IMAGES_PATH, exist_ok=True )
os.makedirs( VALID_LABELS_PATH, exist_ok=True )
os.makedirs( TEST_IMAGES_PATH, exist_ok=True )
os.makedirs( TEST_LABELS_PATH, exist_ok=True )

# Create temporary valid and test folder
TEMP_VALID = os.path.join( dataset_path, 'temp_valid' )
TEMP_TEST = os.path.join( dataset_path, 'temp_test' )
os.makedirs( TEMP_VALID, exist_ok=True )
os.makedirs( TEMP_TEST, exist_ok=True )

## Restruction and Augmentation Dataset

In [ ]:
# Function to check if the number in the filename is odd or even
def is_odd(filename):
    match = re.search(r'\((\d+)\)\.\w+$', filename)
    if match:
        return int(match.group(1)) % 2 == 1
    return False

# Function to copy files while preserving subfolder structure
def temp_valid_test_division(source_path, temp_valid_path, temp_test_path):
    for subfolder in os.listdir(source_path):
        subfolder_path = os.path.join(source_path, subfolder)

        if os.path.isdir(subfolder_path):
            # Create the same subfolder structure in the destination paths
            valid_subfolder = os.path.join(temp_valid_path, subfolder)
            test_subfolder = os.path.join(temp_test_path, subfolder)

            os.makedirs(valid_subfolder, exist_ok=True)
            os.makedirs(test_subfolder, exist_ok=True)

            for file in os.listdir(subfolder_path):
                source_file = os.path.join(subfolder_path, file)

                if os.path.isfile(source_file):
                    dest_folder = valid_subfolder if is_odd(file) else test_subfolder
                    dest_file = os.path.join(dest_folder, file)

                    if not os.path.exists(dest_file):  # Check if file already exists
                        shutil.copy2(source_file, dest_file)
                        print(f'Copied {source_file} -> {dest_file}')
                    else:
                        print(f'Skipped {source_file}, already exists in destination.')

In [ ]:
# Run division function on TEST data (TEST --> VALID & TEST)
temp_valid_test_division(test_path, TEMP_VALID, TEMP_TEST)
print("✅ Division on Test data was complete!")

In [ ]:
# YOLO annotation reading function
def read_yolo_annotation( file_path: str ) -> list:
    annotations = []
    with open( file_path, 'r' ) as f:
        for line in f.readlines():
            parts = line.strip().split()
            class_id = int( parts[0] )
            cx, cy, w, h = map( float, parts[1:] )
            annotations.append( (class_id, cx, cy, w, h) )
    return annotations

# YOLO annotation saving function
def save_yolo_annotation( file_path: str, annotations: list ) -> None:
    with open( file_path, 'w' ) as f:
        for annotation in annotations:
            class_id, cx, cy, w, h = annotation
            f.write( f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n" )  # f.write(f"{class_id} {cx} {cy} {w} {h}\n")

In [ ]:
# Rotate the image and annotations
def rotate_image_and_annotations(image, annotations: list, angle: float) -> tuple:
    # Get image dimensions
    (h, w) = image.shape[:2]
    center = (w / 2, h / 2)

    # Create rotation matrix
    matrix = cv2.getRotationMatrix2D(center, -angle, 1.0)

    # Apply rotation to the image
    rotated_image = cv2.warpAffine(image, matrix, (w, h), flags=cv2.INTER_LINEAR)

    # Convert angle to radians and compute sine and cosine values
    angle_rad = radians(angle)
    sin_angle, cos_angle = sin(angle_rad), cos(angle_rad)

    # Process rotation for each annotation
    rotated_annotations = []
    for annotation in annotations:
        class_id, cx, cy, bw, bh = annotation

        # Convert normalized coordinates to pixel coordinates
        cx_pixel, cy_pixel = cx * w, cy * h
        bw_pixel, bh_pixel = bw * w, bh * h

        # Rotate bounding box center
        new_cx_pixel = center[0] + (cx_pixel - center[0]) * cos_angle - (cy_pixel - center[1]) * sin_angle
        new_cy_pixel = center[1] + (cx_pixel - center[0]) * sin_angle + (cy_pixel - center[1]) * cos_angle

        # Adjust bounding box size after rotation
        new_bw_pixel = abs(bh_pixel * sin_angle) + abs(bw_pixel * cos_angle)
        new_bh_pixel = abs(bh_pixel * cos_angle) + abs(bw_pixel * sin_angle)

        # Normalize rotated coordinates
        new_cx, new_cy = new_cx_pixel / w, new_cy_pixel / h
        new_bw, new_bh = new_bw_pixel / w, new_bh_pixel / h

        # Store rotated annotation
        rotated_annotations.append((class_id, new_cx, new_cy, new_bw, new_bh))

    return rotated_image, rotated_annotations

# Night effect simulation function with contrast adjustment
def apply_night_effect(image, contrast_range=(-0.2, 0.2)):
    pil_img = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    # Adjust brightness (darken image)
    brightness_enhancer = ImageEnhance.Brightness(pil_img)
    darkened = brightness_enhancer.enhance(0.3)  # Reduce brightness by 70%

    # Adjust contrast randomly within the range (-20% to +20%)
    contrast_factor = 1 + np.random.uniform(contrast_range[0], contrast_range[1])
    contrast_enhancer = ImageEnhance.Contrast(darkened)
    final_image = contrast_enhancer.enhance(contrast_factor)

    return cv2.cvtColor(np.array(final_image), cv2.COLOR_RGB2BGR)

In [ ]:
# Ensure the output folder exists before saving files
def ensure_folder_exists(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)

# Dataset augmentation function
def augment_dataset(image_folder, output_image_folder, output_label_folder):
    image_paths = glob.glob(os.path.join(image_folder, "**", "*.jpg"), recursive=True)

    if not image_paths:
        print("🚨 No images found for augmentation!")
        return

    ensure_folder_exists(output_image_folder)
    ensure_folder_exists(output_label_folder)

    for img_path in image_paths:
        img_name = os.path.basename(img_path)
        img_subfolder = os.path.dirname(img_path)  # Get the image's subfolder
        label_path = os.path.join(img_subfolder, img_name.replace(".jpg", ".txt"))

        print(f"🔍 Processing: {img_name}")

        # Copy original image
        original_img_dest = os.path.join(output_image_folder, img_name)
        if not os.path.exists(original_img_dest):
            shutil.copy2(img_path, original_img_dest)
            print(f"✅ Copied original image: {img_name}")

        # Copy original label
        original_label_dest = os.path.join(output_label_folder, os.path.basename(label_path))
        if os.path.exists(label_path):
            if not os.path.exists(original_label_dest):
                shutil.copy2(label_path, original_label_dest)
                print(f"✅ Copied original label: {os.path.basename(label_path)}")
        else:
            print(f"⚠ No label file for {img_name}, skipping.")
            continue

        # Output file paths (without subfolders)
        rotated_img_path = os.path.join(output_image_folder, img_name.replace(".jpg", "_augrotate.jpg"))
        rotated_label_path = os.path.join(output_label_folder, img_name.replace(".jpg", "_augrotate.txt"))
        night_img_path = os.path.join(output_image_folder, img_name.replace(".jpg", "_augnight.jpg"))
        night_label_path = os.path.join(output_label_folder, img_name.replace(".jpg", "_augnight.txt"))

        # Determine if augmentation is needed
        need_rotation = not (os.path.exists(rotated_img_path) and os.path.exists(rotated_label_path))
        need_night_effect = not (os.path.exists(night_img_path) and os.path.exists(night_label_path))

        if not need_rotation and not need_night_effect:
            print(f"⏩ {img_name} already has complete augmentations, skipping.")
            continue

        print(f"🛠 Augmenting {img_name}...")

        # Read the image and annotations
        image = cv2.imread(img_path)
        annotations = read_yolo_annotation(label_path)

        # Rotation augmentation
        if need_rotation:
            angle = random.uniform(-15, 15)
            rotated_image, rotated_annotations = rotate_image_and_annotations(image, annotations, angle)
            cv2.imwrite(rotated_img_path, rotated_image)
            save_yolo_annotation(rotated_label_path, rotated_annotations)
            print(f"  🔄 Rotation {angle:.2f}° saved.")

        # Night effect augmentation
        if need_night_effect:
            night_image = apply_night_effect(image)
            cv2.imwrite(night_img_path, night_image)
            save_yolo_annotation(night_label_path, annotations)
            print("  🌙 Night effect saved.")

In [ ]:
# TEST FUNCTION - Function to draw YOLO bounding boxes on an image
def draw_yolo_bboxes(image_path: str, annotation_path: str):
    image = cv2.imread(image_path)
    if image is None:
        print(f"🚨 Error: Unable to load image {image_path}")
        return

    h, w, _ = image.shape  # Get image dimensions

    annotations = read_yolo_annotation(annotation_path)

    for annotation in annotations:
        class_id, cx, cy, bw, bh = annotation

        # Convert YOLO format to pixel coordinates
        x1 = int((cx - bw / 2) * w)
        y1 = int((cy - bh / 2) * h)
        x2 = int((cx + bw / 2) * w)
        y2 = int((cy + bh / 2) * h)

        # Draw bounding box
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, str(class_id), (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Convert BGR to RGB for displaying
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Display the image
    plt.figure(figsize=(8, 6))
    plt.imshow(image_rgb)
    plt.axis("on")
    plt.show()

In [ ]:
# # TEST FUNCTION
# def test_single_image(image_path, label_path):
#     if not os.path.exists(image_path) or not os.path.exists(label_path):
#         print("🚨 Error: Image or label file not found!")
#         return

#     # Read image and annotations
#     image = cv2.imread(image_path)
#     annotations = read_yolo_annotation(label_path)

#     # Display original image with bounding boxes
#     print("📌 Original Image")
#     draw_yolo_bboxes(image_path, label_path)

#     # Apply rotation
#     angle = random.uniform(-15, 15)
#     rotated_image, rotated_annotations = rotate_image_and_boxes(image, annotations, angle)

#     # Save and display rotated image
#     rotated_img_path = image_path.replace(".jpg", "_testrotate.jpg")
#     rotated_label_path = label_path.replace(".txt", "_testrotate.txt")
#     cv2.imwrite(rotated_img_path, rotated_image)
#     save_yolo_annotation(rotated_label_path, rotated_annotations)
#     print(f"🔄 Rotated {angle:.2f}°")
#     draw_yolo_bboxes(rotated_img_path, rotated_label_path)

#     # Apply night effect
#     night_image = apply_night_effect(image)

#     # Save and display night effect image
#     night_img_path = image_path.replace(".jpg", "_testnight.jpg")
#     night_label_path = label_path.replace(".txt", "_testnight.txt")
#     cv2.imwrite(night_img_path, night_image)
#     save_yolo_annotation(night_label_path, annotations)  # Label stays the same
#     print("🌙 Night Effect Applied")
#     draw_yolo_bboxes(night_img_path, night_label_path)

# # test with one image
# test_image_path = "/content/drive/MyDrive/Datasets/indonesian-traffic-sign/train/lampu-hijau/lampu hijau (43).jpg"
# test_label_path = "/content/drive/MyDrive/Datasets/indonesian-traffic-sign/train/lampu-hijau/lampu hijau (43).txt"
# test_single_image(test_image_path, test_label_path)

In [ ]:
# Run augmentation for the TRAIN dataset
augment_dataset( train_path, TRAIN_IMAGES_PATH, TRAIN_LABELS_PATH )
print("✅ Augmentation on Train data was complete!")

In [ ]:
# Paths to augmented images and labels
path_image = TRAIN_IMAGES_PATH
path_label = TRAIN_LABELS_PATH
class_name = 'lampu hijau'
image_number = 7 # Choose number between 1 - 70

image_path = os.path.join( path_image, f'{class_name} ({image_number}).jpg' )
label_path = os.path.join( path_label, f'{class_name} ({image_number}).txt' )
image1_path = os.path.join( path_image, f'{class_name} ({image_number})_augnight.jpg' )
label1_path = os.path.join( path_label, f'{class_name} ({image_number})_augnight.txt' )
image2_path = os.path.join( path_image, f'{class_name} ({image_number})_augrotate.jpg' )
label2_path = os.path.join( path_label, f'{class_name} ({image_number})_augrotate.txt' )

# Display images with bounding boxes
draw_yolo_bboxes(image_path, label_path)
draw_yolo_bboxes(image1_path, label1_path)
draw_yolo_bboxes(image2_path, label2_path)

> **Train dataset is ready to use**


In [ ]:
# Run augmentation for the VALID dataset
augment_dataset( TEMP_VALID, VALID_IMAGES_PATH, VALID_LABELS_PATH )
print("✅ Augmentation on Valid data was complete!")

In [ ]:
# Paths to augmented images and labels
path_image = VALID_IMAGES_PATH
path_label = VALID_LABELS_PATH
class_name = 'lampu hijau'
image_number = 87 # Choose odd number between 71 - 100

image_path = os.path.join( path_image, f'{class_name} ({image_number}).jpg' )
label_path = os.path.join( path_label, f'{class_name} ({image_number}).txt' )
image1_path = os.path.join( path_image, f'{class_name} ({image_number})_augnight.jpg' )
label1_path = os.path.join( path_label, f'{class_name} ({image_number})_augnight.txt' )
image2_path = os.path.join( path_image, f'{class_name} ({image_number})_augrotate.jpg' )
label2_path = os.path.join( path_label, f'{class_name} ({image_number})_augrotate.txt' )

# Display images with bounding boxes
draw_yolo_bboxes(image_path, label_path)
draw_yolo_bboxes(image1_path, label1_path)
draw_yolo_bboxes(image2_path, label2_path)

> **Valid dataset is ready to use**

In [ ]:
# Run augmentation for the TEST dataset
augment_dataset( TEMP_TEST, TEST_IMAGES_PATH, TEST_LABELS_PATH )
print("✅ Augmentation on Test data was complete!")

In [ ]:
# Paths to augmented images and labels
path_image = TEST_IMAGES_PATH
path_label = TEST_LABELS_PATH
class_name = 'lampu hijau'
image_number = 94 # Choose even number between 71 - 100

image_path = os.path.join( path_image, f'{class_name} ({image_number}).jpg' )
label_path = os.path.join( path_label, f'{class_name} ({image_number}).txt' )
image1_path = os.path.join( path_image, f'{class_name} ({image_number})_augnight.jpg' )
label1_path = os.path.join( path_label, f'{class_name} ({image_number})_augnight.txt' )
image2_path = os.path.join( path_image, f'{class_name} ({image_number})_augrotate.jpg' )
label2_path = os.path.join( path_label, f'{class_name} ({image_number})_augrotate.txt' )

# Display images with bounding boxes
draw_yolo_bboxes(image_path, label_path)
draw_yolo_bboxes(image1_path, label1_path)
draw_yolo_bboxes(image2_path, label2_path)

> **Test dataset is ready to use**

In [ ]:
# Count data for each dataset

# Train
data = os.listdir( TRAIN_IMAGES_PATH )
count = len( data )
print( 'Train images :', count )
data = os.listdir( TRAIN_LABELS_PATH )
count = len( data )
print( 'Train labels :', count, '\n' )

# Validation
data = os.listdir( VALID_IMAGES_PATH )
count = len( data )
print( 'Valid images :', count )
data = os.listdir( VALID_LABELS_PATH )
count = len( data )
print( 'Valid labels :', count, '\n' )

# Test
data = os.listdir( TEST_IMAGES_PATH )
count = len( data )
print( 'Test images  :', count )
data = os.listdir( TEST_LABELS_PATH )
count = len( data )
print( 'Test labels  :', count )



> I'm quite happy with this number, it's ideal



## Train the dataset using YOLO

In [ ]:
# Build a model detection using YOLOv8

# Load a model
model = YOLO("yolov8n.yaml") # build a new model from scratch

config_path = os.path.join( dataset_path, 'config.yaml' )

# Train the model on the COCO8 example dataset for 100 epochs
results = model.train( data=config_path, epochs=64, imgsz=416 )